In [ ]:
from backtesting import Backtest, Strategy
import pandas as pd
from backtesting.lib import crossover, plot_heatmaps, resample_apply
import seaborn as sns
import matplotlib.pyplot as plt
import mpld3
import numpy as np
import time
from talib import CDLENGULFING, ADX, CCI, ATR
from DataPaths import data_paths
import plotly.express as px


In [ ]:
def load_and_prepare_data(file_path, start_date, end_date):
    # Load CSV
    data = pd.read_csv(file_path, parse_dates=['Time'], index_col='Time')
    # Ensure index is datetime
    data.index = pd.to_datetime(data.index)

    # Print column names to verify
    print("Columns in the CSV file:", data.columns)

    # Select required columns
    data = data[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

    # Remove duplicate indexes
    if data.index.duplicated().any():
        print("Duplicate indexes found. Removing duplicates.")
        data = data[~data.index.duplicated(keep='first')]

    # Check for NaN values
    print("Checking for NaN values in the data:")
    print(data.isna().sum())

    # Drop NaN values
    data = data.dropna()

    # Reduce the number of data points to a specific date range
    data = data.loc[start_date:end_date]

    print(f"Number of data points after reduction: {len(data)}")

    return data


In [ ]:
if __name__ == '__main__':

    file_path = data_paths["NAS100"]["H1"]
    
    start_date = '2020-01-01'
    end_date = '2021-01-01'

    data = load_and_prepare_data(file_path, start_date, end_date)

    print("loading backtest results.....")
    
    bt = Backtest(data, SimpleEngulfingStrategy, cash=100_000)
    
    stats = bt.run()
    print(stats)
    bt.plot()
    

In [ ]:
# Save the trades to a CSV file in a folder named "results"
import os

# Create the folder if it doesn't exist
output_folder = "results"
os.makedirs(output_folder, exist_ok=True)

# Define the full file path
output_file = os.path.join(output_folder, "trades.csv")

# Save the DataFrame to CSV
stats["_trades"].to_csv(output_file, index=False)  # index=False to skip row numbers in CSV
